# Notebook 06 — Hypothesis-tailored BFT on ImageNet ("dataset probes")

Standard BFT (nb05) factorizes the joint arbor matrices over a **broad, category-balanced** stimulus set — the SAE end of the interpretability spectrum: unsupervised, rank budget spent on the dominant global structure. This notebook moves BFT toward the **linear-probe / TCAV** end *without changing the method*: the stimulus set itself is the probe. Enrich the set with a hypothesis (all bear species, one dog breed group, "striped things") and the factorization's reconstruction budget is reallocated onto the circuitry that processes those stimuli — sub-structure the broad trace blurs into a single factor becomes resolvable.

The trace stays fully unsupervised — the *only* supervision is the choice of stimuli. Group labels in a hypothesis encode the **expected** sub-structure and are used purely for evaluation and coloring.

**Usage** — pick a hypothesis and run top to bottom:
```bash
TAILORED_HYP=dogs python scripts/run_nb.py notebooks/06_tailored_imagenet.ipynb
```
or edit `HYP_NAME` in §1. Build your own with `find_imagenet_classes('terrier')`. Optional deep-dives are env-gated: `TAILORED_RANKS=1` (held-out rank re-derivation), `TAILORED_DOSE=1` (enrichment dose–response), `TAILORED_PRUNE=1` (causal pruning selectivity).

Sections: §0 setup · §1 hypothesis registry · §2 data · §3 trace · §4 factor exploration · §5 sub-structure vs baselines · §6 correspondence with the broad (nb05) tree · §7 dose–response · §8 causal check. Research agenda: `TAILORED_BFT.md` at the repo root.

Requires the ImageNet val split at `data/val` (ImageFolder layout), like nb05.

## §0 — Imports & setup

In [ ]:
#%matplotlib inline
import sys, os, glob, pickle, time
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from torch.utils.data import DataLoader, Subset

from src import (
    bft, cached_tree, cached_result, truncate_tree,
    extract_tree_nodes, nodes_at_layer, extract_fingerprint_matrix,
    project_stimuli_onto_tree, plot_factor_overview_panel, plot_factor_gallery,
    imdenorm as _imdenorm,
    # tailored-BFT module (this branch)
    imagenet_class_names, find_imagenet_classes, collect_hypothesis_data,
    subsample_data, mixture_indices, factor_label_profile, substructure_scores,
    match_factors, refinement_score,
)
from src.ablation_utils import select_class_circuit, run_ablation_sweep

plt.rcParams.update({'figure.dpi': 80})
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

## §1 — Hypothesis registry

A hypothesis is a dict: `groups` maps group names to lists of ImageNet class ids (`find_imagenet_classes` to look ids up); `n_per_class` caps stimuli per fine class (val has ~50); `k_root` sets the output-layer NMF rank; `prune_group` names the §8 target. Groups never enter the trace — they define what sub-structure you *expect*, for evaluation.

In [ ]:
# ── Paths & constants (as nb05) ───────────────────────────────────────────────
IMAGENET_DIR  = '../data'
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_SIZE      = 224

def squeezenet_spine_filter(name, mod):
    """The sequential squeeze-spine: initial conv, squeeze convs, classifier conv."""
    return (name == 'features.0' or
            name == 'classifier.1' or
            (isinstance(mod, nn.Conv2d) and name.endswith('.squeeze')))

# ── Hypothesis registry (all ids verified against torchvision class names) ────
HYPOTHESES = {
    # Species zoom below nb05's 'bear' category — the regime where BFT beat
    # activation baselines in the error-consistency pilot study.
    'bears': dict(
        groups={'brown_bear': [294], 'black_bear': [295], 'ice_bear': [296],
                'sloth_bear': [297], 'lesser_panda': [387], 'giant_panda': [388]},
        n_per_class=50, k_root=8, prune_group='ice_bear'),
    # Breed-group structure inside the dog super-category (nb05 saw 2 dog classes).
    'dogs': dict(
        groups={'terriers':   [181, 182, 184, 187],
                'retrievers': [205, 207, 208, 219],
                'working':    [235, 248, 249, 250],
                'sheepdogs':  [229, 230, 231, 232]},
        n_per_class=40, k_root=10, prune_group='terriers'),
    # One perceptual family across two output regimes: domestic vs big cats.
    'cats': dict(
        groups={'domestic': [281, 282, 283, 284, 285],
                'big':      [286, 288, 290, 291, 292, 293]},
        n_per_class=40, k_root=8, prune_group='big'),
    # Three canid families — does the trace split by taxonomy or appearance?
    'canids': dict(
        groups={'dogs':   [231, 235, 249, 250],
                'wolves': [269, 270, 271],
                'foxes':  [277, 278, 279, 280]},
        n_per_class=40, k_root=8, prune_group='foxes'),
    # Habitat split inside the bird category.
    'birds': dict(
        groups={'songbirds':  [10, 11, 13, 15, 17, 18, 19],
                'waterbirds': [99, 130, 131, 144, 145, 146]},
        n_per_class=40, k_root=8, prune_group='waterbirds'),
    # Attribute probe crossing categories: striped vs matched plain counterparts
    # (zebra~sorrel, tiger~lion, tiger cat~Egyptian cat). Any factor unify stripes?
    'striped': dict(
        groups={'striped': [340, 292, 282],
                'plain':   [339, 291, 285]},
        n_per_class=50, k_root=6, prune_group='striped'),
    # Null control: arbitrary classes in arbitrary groups — calibrates how much
    # "sub-structure" NMF invents on sets with no real shared circuitry.
    'random_control': dict(
        groups={f'group{g}': ids.tolist() for g, ids in enumerate(
            np.random.default_rng(0).choice(1000, 12, replace=False).reshape(3, 4))},
        n_per_class=40, k_root=8, prune_group='group0'),
}

# ── Selection & flags ─────────────────────────────────────────────────────────
HYP_NAME       = os.environ.get('TAILORED_HYP', 'bears')
CORRECTNESS    = os.environ.get('TAILORED_CORRECT', 'strict')  # strict | group | none
REDERIVE_RANKS = bool(int(os.environ.get('TAILORED_RANKS', '0')))
RUN_DOSE       = bool(int(os.environ.get('TAILORED_DOSE',  '0')))
RUN_PRUNE      = bool(int(os.environ.get('TAILORED_PRUNE', '0')))

HYP = dict(HYPOTHESES[HYP_NAME], name=HYP_NAME)
GROUPS      = HYP['groups']
GROUP_NAMES = list(GROUPS)
N_GROUPS    = len(GROUPS)
HYP_IDS     = sorted({c for ids in GROUPS.values() for c in ids})
K_ROOT      = HYP.get('k_root') or max(6, min(16, 2 * N_GROUPS))

# Default per-layer rank profile: nb05's selected spine profile with the output
# rank swapped for the hypothesis's K_ROOT (10 spine layers). §3b can re-derive
# ranks on the tailored arbors (TAILORED_RANKS=1).
K_MAX      = [6, 5, 7, 6, 7, 6, 6, 4, 6, K_ROOT]
N_BRANCHES = [1, 1, 1, 1, 1, 1, 1, 1, 2, K_ROOT]

_names = imagenet_class_names()
print(f"hypothesis '{HYP_NAME}': {N_GROUPS} groups, {len(HYP_IDS)} fine classes, "
      f"correctness='{CORRECTNESS}'")
for g, ids in GROUPS.items():
    print(f'  {g:<12} {[f"{c}:{_names[c]}" for c in ids]}')
print(f'K_MAX={K_MAX}  N_BRANCHES={N_BRANCHES}')
# Build your own: find_imagenet_classes('husky')

## §2 — Data: tailored stimulus set → spine layer dicts

`collect_hypothesis_data` restricts the val split to the hypothesis classes, runs the model, keeps only samples passing the correctness filter (`strict` = 1000-way top-1 correct; `group` = predicted *some* hypothesis class, i.e. the model ran the right circuitry even if it missed the species; `none`), and balances per fine class. Fine ImageNet labels are kept in `data['fine']` — all sub-structure metrics use them.

In [ ]:
# ── Val split + pretrained SqueezeNet (as nb05) ───────────────────────────────
normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
test_tfm  = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor(), normalize])

def _make_imagenet(split, transform):
    try:
        return torchvision.datasets.ImageNet(IMAGENET_DIR, split=split, transform=transform)
    except Exception:
        folder = 'train' if split == 'train' else 'val'
        return torchvision.datasets.ImageFolder(
            os.path.join(IMAGENET_DIR, folder), transform=transform)

val_ds = _make_imagenet('val', test_tfm)
model  = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE)
model.eval()
print(f'val: {len(val_ds):,} images | SqueezeNet 1.1: '
      f'{sum(p.numel() for p in model.parameters()):,} params')

t0 = time.time()
data = collect_hypothesis_data(model, val_ds, HYP, DEVICE, squeezenet_spine_filter,
                               n_per_class=HYP.get('n_per_class', 40),
                               correctness=CORRECTNESS)
print(f'collected in {time.time() - t0:.0f}s')

images       = data['images']
targets      = data['targets']          # group labels 0..G-1
fine         = data['fine']             # original ImageNet ids
layer_inputs = [ld['input_fmap'] for ld in data['layer_data']]
layer_names  = [ld['name'] for ld in data['layer_data']]
n_samples    = len(images)
imdenorm     = lambda img: _imdenorm(img, IMAGENET_MEAN, IMAGENET_STD)
FINE_ORDER   = [c for ids in GROUPS.values() for c in ids if (fine == c).any()]
print(f'{n_samples} stimuli | {len(layer_inputs)} spine layers: {layer_names}')

## §3 — Tailored trace

### 3a — BFT on the hypothesis set (cached per hypothesis + HPs)

In [ ]:
tree = cached_tree(f'nb06_{HYP_NAME}_circuit', lambda: bft(
    data['layer_data'],
    k_max=K_MAX, n_branches=N_BRANCHES,
    conv_pool_method='avg', stimulus_threshold=0.0,
    weighting='img_selectivity', verbose=1, n_jobs=3,
), params=dict(k=K_MAX, b=N_BRANCHES, n=n_samples, corr=CORRECTNESS,
               npc=HYP.get('n_per_class', 40)))

tree_nodes = extract_tree_nodes(tree)
K_root_eff = len(tree.root.lambdas)
print(f'tree nodes: {len(tree_nodes)}   K_root: {K_root_eff}')

### 3b — Optional: held-out rank re-derivation on the tailored arbors (`TAILORED_RANKS=1`)

If enrichment reveals latent sub-structure, the held-out criterion should ask for *more* rank than the broad trace did at the same layers — that shift is itself evidence. Prints the suggested profile; re-trace by putting it into §1 if it differs materially.

In [ ]:
if REDERIVE_RANKS:
    from src import node_pos_arbor, nodes_per_layer, select_ranks
    _npl = nodes_per_layer(tree, max_nodes=2)
    _arbors = {li: [node_pos_arbor(nd, layer_inputs[li]) for nd in nds]
               for li, nds in _npl.items()}
    hp_sel = cached_result(
        f'nb06_{HYP_NAME}_hpsel',
        lambda: select_ranks(_arbors, targets, k_cap=16,
                             n_classes=N_GROUPS, last_extra=4),
        params=dict(kcap=16, n=n_samples, k=K_MAX))
    print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
    print('assembled profile     k_max=%s  n_branches=%s'
          % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
    print('§1 default was        k_max=%s' % (K_MAX,))
else:
    print('skipped (TAILORED_RANKS=1 to run)')

## §4 — Explore the tailored factors

### 4a — Factor overview panels (root + level-1 nodes)

In [ ]:
for node in tree_nodes:
    if len(node['path']) > 1:
        continue                      # root + its immediate sub-circuits only
    label = 'root' if not node['path'] else 'F' + '-F'.join(map(str, node['path']))
    print(f"── {node['layer_name']} [{label}] "
          f"K={node['img_factors'].shape[1]} " + '─' * 40)
    figs = plot_factor_overview_panel(node, images, targets, data['class_names'])
    for f in figs:
        plt.show()

### 4b — Fine-class profile per root factor

The central readout: how each root factor's loading mass distributes over the *fine* ImageNet classes. Block structure along the expected groups = the hypothesis's sub-structure is what the factorization found; a factor with high purity on one species/breed is a discovered sub-circuit.

In [ ]:
prof = factor_label_profile(tree.root.img_factors, fine)
_col = [list(prof['labels']).index(c) for c in FINE_ORDER]
M = prof['mass'][:, _col]

fig, ax = plt.subplots(figsize=(max(6, 0.45 * len(FINE_ORDER)), 0.45 * len(M) + 1.5))
im = ax.imshow(M, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(FINE_ORDER)))
ax.set_xticklabels([_names[c] for c in FINE_ORDER], rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(len(M)))
ax.set_yticklabels([f'F{k}  (purity {prof["purity"][k]:.2f})' for k in range(len(M))],
                   fontsize=8)
_gs = np.cumsum([sum((fine == c).any() for c in ids) for ids in GROUPS.values()])[:-1]
for x in _gs:
    ax.axvline(x - 0.5, color='w', lw=1)
ax.set_title(f'{HYP_NAME}: root-factor loading mass over fine classes')
plt.colorbar(im, ax=ax, label='share of factor mass')
plt.tight_layout(); plt.show()

print('factor purity  :', np.round(prof['purity'], 2))
print('factor entropy :', np.round(prof['entropy'], 2), '(0 = pure)')
gprof = factor_label_profile(tree.root.img_factors, targets)
print('group purity   :', np.round(gprof['purity'], 2),
      ' dominant group:', [GROUP_NAMES[j] for j in gprof['mass'].argmax(1)])

## §5 — Quantify: does tailoring resolve sub-structure the broad trace misses?

Four representations of the *same* tailored stimuli, scored on how well they separate the fine classes (and the groups):

1. **tailored fingerprints** — top-2 slice of the tailored tree (this notebook)
2. **broad fingerprints** — the same stimuli NNLS-projected onto the *fixed* nb05 broad-tree factors (the "SAE dictionary" baseline; skipped if the nb05 cache is absent)
3. **raw activations, penultimate** (spatially pooled) — the baseline that subsumed BFT on target-aligned structure in the error-consistency study; tailoring earns its keep only where it beats this
4. **raw activations, all spine layers** (pooled, concatenated)

In [ ]:
def _load_broad_tree():
    """Newest cached nb05 circuit tree, or None."""
    hits = sorted(glob.glob('../data/cache/nb05_circuit__*.pkl'), key=os.path.getmtime)
    if not hits:
        return None
    with open(hits[-1], 'rb') as f:
        obj = pickle.load(f)
    print(f'broad tree: {os.path.basename(hits[-1])}')
    return obj

broad_tree = _load_broad_tree()
_pool = lambda a: a.mean(axis=(2, 3)) if a.ndim == 4 else a.reshape(len(a), -1)

reps = {}
F_tail = extract_fingerprint_matrix(truncate_tree(tree, depth=2), np.arange(n_samples))
reps['tailored fp (top-2)'] = F_tail
if broad_tree is not None:
    proj_broad_top2 = project_stimuli_onto_tree(truncate_tree(broad_tree, depth=2),
                                                layer_inputs)
    reps['broad fp (NNLS)'] = extract_fingerprint_matrix(proj_broad_top2,
                                                         np.arange(n_samples))
reps['act penultimate'] = _pool(layer_inputs[-1])
reps['act all layers']  = np.concatenate([_pool(x) for x in layer_inputs], axis=1)

print(f'{"representation":<20} | {"fine: sil":>9} {"knn":>6} {"ari":>6} '
      f'| {"group: sil":>10} {"knn":>6} {"ari":>6} |    d')
for name, F in reps.items():
    sf = substructure_scores(F, fine)
    sg = substructure_scores(F, targets)
    print(f'{name:<20} | {sf["silhouette"]:>9.3f} {sf["knn_acc"]:>6.3f} '
          f'{sf["kmeans_ari"]:>6.3f} | {sg["silhouette"]:>10.3f} '
          f'{sg["knn_acc"]:>6.3f} {sg["kmeans_ari"]:>6.3f} | {sf["d"]:>4d}')

In [ ]:
# ── PCA embeddings colored by fine class ──────────────────────────────────────
from sklearn.decomposition import PCA
_show = [k for k in ['tailored fp (top-2)', 'broad fp (NNLS)', 'act penultimate']
         if k in reps]
fig, axes = plt.subplots(1, len(_show), figsize=(5 * len(_show), 4.4))
axes = np.atleast_1d(axes)
cmap = plt.get_cmap('tab20')
for ax, name in zip(axes, _show):
    Z = PCA(n_components=2).fit_transform(reps[name])
    for j, c in enumerate(FINE_ORDER):
        m = fine == c
        ax.scatter(Z[m, 0], Z[m, 1], s=10, color=cmap(j % 20),
                   label=_names[c], alpha=0.75)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[-1].legend(fontsize=7, markerscale=1.5, loc='center left',
                bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(); plt.show()

## §6 — Correspondence with the broad tree: refinement or rotation?

Cosine-match the tailored root factors against the broad (nb05) root factors in the shared arbor space. **Refinement** (each tailored factor descends from one broad factor — typically the hypothesis category's factor splitting into sub-circuits) supports "tailoring = zoom". **Rotation** (tailored factors mix several broad factors) means the enriched distribution changed the basis itself — interesting, but the zoom interpretation weakens.

In [ ]:
if broad_tree is None:
    print('nb05 broad tree cache not found — run nb05 first to enable §6.')
else:
    corr = match_factors(tree.root,
                         broad_tree.root if hasattr(broad_tree, 'root') else broad_tree)
    per, mean_ref = refinement_score(corr['sim'])

    fig, ax = plt.subplots(figsize=(0.5 * corr['sim'].shape[1] + 2,
                                    0.45 * corr['sim'].shape[0] + 1.5))
    im = ax.imshow(corr['sim'], cmap='magma', vmin=0)
    ax.set_xlabel('broad root factors (nb05)')
    ax.set_ylabel(f'tailored root factors ({HYP_NAME})')
    ax.set_title(f'connection-factor cosine — refinement score {mean_ref:.2f} '
                 '(1 = clean refinement, 0 = rotation)')
    plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

    # Which group each broad factor prefers, measured on the tailored stimuli
    # via the NNLS projection (self-contained — no nb05 stimuli needed).
    proj_full = project_stimuli_onto_tree(broad_tree, layer_inputs)
    bprof = factor_label_profile(proj_full.img_factors, targets)
    for k in range(corr['sim'].shape[0]):
        j = corr['match'][k]
        print(f'tailored F{k}  <-  broad F{j}  (cos {corr["sim"][k, j]:.2f}, '
              f'margin {per[k]:.2f}); broad F{j} prefers '
              f'"{GROUP_NAMES[bprof["mass"][j].argmax()]}" '
              f'({bprof["mass"][j].max():.2f} of its mass here)')

## §7 — Dose–response: how much enrichment does sub-structure need? (`TAILORED_DOSE=1`)

Total N fixed; the fraction ρ of hypothesis stimuli sweeps from the broad-mix regime to the pure hypothesis set (the rest drawn from nb05's 16 reference classes). The rank profile stays fixed across ρ so emergence is attributable to the stimulus distribution, not to rank re-selection. Readout: fine-class separability of the hypothesis stimuli, from each mixture-trace's fingerprints.

In [ ]:
if RUN_DOSE:
    REF_IDS = [c for c in [404, 895, 403, 724, 609, 751, 444, 671,
                           101, 385, 294, 297, 151, 251, 7, 9]      # nb05 focus
               if c not in HYP_IDS]
    mix_hyp = dict(name=f'{HYP_NAME}+ref',
                   groups={**GROUPS, '_ref': REF_IDS})
    pool = collect_hypothesis_data(model, val_ds, mix_hyp, DEVICE,
                                   squeezenet_spine_filter,
                                   n_per_class=HYP.get('n_per_class', 40),
                                   correctness=CORRECTNESS)
    n_hyp_avail = int(np.isin(pool['fine'], HYP_IDS).sum())
    N_TOTAL = min(800, n_hyp_avail)
    print(f'pool: {len(pool["fine"])} stimuli, {n_hyp_avail} hypothesis, '
          f'N_TOTAL={N_TOTAL}')

    RHOS, dose = [0.125, 0.25, 0.5, 0.75, 1.0], []
    for rho in RHOS:
        idx = mixture_indices(pool['fine'], HYP_IDS, REF_IDS, rho, N_TOTAL, seed=0)
        sub = subsample_data(pool, idx)
        t = cached_tree(f'nb06_{HYP_NAME}_dose', lambda: bft(
                sub['layer_data'], k_max=K_MAX, n_branches=N_BRANCHES,
                conv_pool_method='avg', weighting='img_selectivity',
                verbose=0, n_jobs=3),
            params=dict(rho=rho, n=len(idx), k=K_MAX, corr=CORRECTNESS))
        F = extract_fingerprint_matrix(truncate_tree(t, depth=2),
                                       np.arange(len(idx)))
        hm = np.isin(sub['fine'], HYP_IDS)
        s = substructure_scores(F[hm], sub['fine'][hm])
        dose.append(s)
        print(f'rho={rho:.3f}  n_hyp={int(hm.sum()):>4}  '
              f'sil={s["silhouette"]:.3f}  knn={s["knn_acc"]:.3f}  '
              f'ari={s["kmeans_ari"]:.3f}')

    fig, ax = plt.subplots(figsize=(5, 3.5))
    for key, mk in [('silhouette', 'o'), ('knn_acc', 's'), ('kmeans_ari', '^')]:
        ax.plot(RHOS, [d[key] for d in dose], marker=mk, label=key)
    ax.set_xlabel('hypothesis fraction ρ (N fixed)')
    ax.set_ylabel('fine-class separability of hypothesis stimuli')
    ax.set_title(f'{HYP_NAME}: enrichment dose–response')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('skipped (TAILORED_DOSE=1 to run)')

## §8 — Causal check: is the tailored circuit *the* circuit for its group? (`TAILORED_PRUNE=1`)

Select the root factor most selective for `prune_group` on the tailored tree, extract its weight-importance scores, and ablate top-ranked weights. If the tailored trace found a real sub-circuit, damage should concentrate on the target group (selectivity = target drop − mean off-target drop) and exceed both a random baseline and — where the nb05 cache exists — the *broad* tree's circuit selected for the same group (the broad factors shouldn't resolve the group as precisely).

In [ ]:
if RUN_PRUNE:
    PRUNE_GROUP = GROUP_NAMES.index(HYP.get('prune_group', GROUP_NAMES[0]))
    FRACS, FRAC_STAT = (0.05, 0.1, 0.2), 0.1

    # Eval loader over all val images of the hypothesis classes; labels and
    # 1000-way predictions both map through fine -> group (else -1 = wrong).
    _tbl = torch.full((1000,), -1, dtype=torch.long)
    for g, ids in enumerate(GROUPS.values()):
        for c in ids:
            _tbl[c] = g
    _map = lambda y: _tbl.to(y.device)[y]
    _vt = np.array(val_ds.targets if hasattr(val_ds, 'targets')
                   else [s[1] for s in val_ds.samples])
    eval_loader = DataLoader(Subset(val_ds, np.where(np.isin(_vt, HYP_IDS))[0]),
                             batch_size=64, shuffle=False, num_workers=4)

    def _sweep(scores, method):
        return run_ablation_sweep(model, scores, FRACS, eval_loader, _map,
                                  DEVICE, method=method, n_random_repeats=3,
                                  layer_names=layer_names, pred_transform=_map)

    def _drops(res, base):
        return {g: {f: base[f][g] - res[f][g] for f in FRACS}
                for g in res[FRACS[0]]}

    from src.ablation_utils import per_class_accuracy
    base = per_class_accuracy(model, eval_loader, _map, DEVICE, pred_transform=_map)
    base = {f: base for f in FRACS}
    print('baseline per-group acc:',
          {GROUP_NAMES[g]: round(a, 3) for g, a in base[FRACS[0]].items()})

    runs = {}
    sc_tail, info = select_class_circuit(tree, targets, PRUNE_GROUP)
    print(f'tailored circuit for "{GROUP_NAMES[PRUNE_GROUP]}": factor '
          f'{info["k_star"]} (selectivity {info["selectivity"]:.2f})')
    runs['tailored circuit'] = _sweep(sc_tail, 'algo_top')
    runs['random'] = _sweep(sc_tail, 'random')
    if broad_tree is not None:
        proj_full = project_stimuli_onto_tree(broad_tree, layer_inputs)
        sc_broad, binfo = select_class_circuit(proj_full, targets, PRUNE_GROUP)
        print(f'broad circuit: factor {binfo["k_star"]} '
              f'(selectivity {binfo["selectivity"]:.2f}, '
              f'is_selective={binfo["is_selective"]})')
        runs['broad circuit'] = _sweep(sc_broad, 'algo_top')

    print(f'\naccuracy drop at fraction {FRAC_STAT}:')
    print(f'{"method":<18} | ' +
          ' '.join(f'{GROUP_NAMES[g][:10]:>10}' for g in sorted(base[FRACS[0]])) +
          ' | selectivity')
    for name, res in runs.items():
        d = _drops(res, base)
        row = [d[g][FRAC_STAT] for g in sorted(d)]
        tgt = d[PRUNE_GROUP][FRAC_STAT]
        off = np.mean([d[g][FRAC_STAT] for g in d if g != PRUNE_GROUP])
        print(f'{name:<18} | ' + ' '.join(f'{v:>+10.3f}' for v in row) +
              f' | {tgt - off:+.3f}')
else:
    print('skipped (TAILORED_PRUNE=1 to run)')